# AOP-ProCon — Step 7: Release package and integration contract

Packages what Component 1 can honestly hand to Components 2–4 and to the examiners:

| Part | What it is | Why it is in the package |
|---|---|---|
| `data/` | S2 benchmark (`AOP-BenchPos_S2.csv`), provenance, conflicts, inactive set, descriptors, the three pair-set modes, manifests (Steps 1–3) | The leak-controlled multi-label benchmark is the most solid result of Component 1 |
| `embeddings/` | ESM-2 35M / 150M / 650M / 3B embeddings of all 1,389 sequences, with row-order files | Reusable by the other components; SHA-256 recorded |
| `probes/` | Two **weak** MC-active-vs-FRS-only probes (`strict`, `extended`) as plain NumPy arrays, plus **out-of-fold scores** for every benchmark peptide | The only supervised mechanism signal; its limits are stated next to it |
| `aop_procon.py` | Small helper: validate sequences, embed with ESM-2 650M (same pooling rule as Step 2), score with the probes, return the contract table | One entry point for teammates |
| `CONTRACT.md` | Input / output specification, versioning, limitations | The "integration contract" asked for in the lecture |
| `README.md`, `RESULTS_SUMMARY.md` | How to use and reproduce; results tables **generated from the Step 5 / Step 6 CSVs** | No hand-typed numbers |
| `models/` | The 15 contrastive checkpoints of Step 4, as research artefacts | Reproducibility only — they do **not** beat the untrained baselines and must not be used for scoring |
| `MANIFEST.json`, `aop_procon_v1.zip` | SHA-256 of every file; one archive | Integrity |

### What is deliberately **not** claimed
The contrastive projection head is not offered as a model: in Step 5 it is worse than untrained ESM-2, amino-acid composition and length-only retrieval. The probes are offered with their repeated-grouped-CV AUROC and its confidence interval, the paired comparison showing they are not distinguishable from amino-acid composition, and the **source / length confound controls** (STEP 08, Step 5 v4): a single binary `source_group` feature is as good as ESM-2 650M, and ESM-2 adds nothing measurable on top of composition + length + source.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE = "/content/drive/MyDrive/AOP-ProCon"

In [ ]:
import os, json, hashlib, itertools, math, datetime
from importlib import metadata
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import wilcoxon
from scipy.special import comb
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

PROC = f"{BASE}/data/processed"; EMB = f"{BASE}/data/embeddings"; RES = f"{BASE}/results"
os.makedirs(RES, exist_ok=True)
device = "cpu"; SEED = 42; N_BOOT = 2000
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)

def _ver(pkg):
    try: return metadata.version(pkg)
    except Exception: return "n/a"
VERSIONS = {"python": __import__("sys").version.split()[0], "torch": torch.__version__, "numpy": np.__version__,
            "pandas": pd.__version__, "scikit-learn": _ver("scikit-learn"), "scipy": _ver("scipy")}
print(VERSIONS)

def s2_fingerprint(s2_df):
    lines = []
    for r in s2_df.sort_values("sequence").itertuples():
        lines.append("\t".join([r.sequence, r.labels,
                                "" if pd.isna(r.homology_group) else str(int(r.homology_group)),
                                "" if pd.isna(r.lso_fold) else str(int(r.lso_fold)),
                                f"{float(r.sample_weight):.1f}"]))
    return hashlib.md5("\n".join(lines).encode()).hexdigest()[:16]

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

In [ ]:
import shutil, zipfile
RELEASE_NAME = "aop_procon_v1"
REL = f"{BASE}/release/{RELEASE_NAME}"
RUN_ESM_SMOKETEST = False       # True on Colab: downloads ESM-2 650M (~2.5 GB) and compares freshly embedded sequences with the cached embeddings

---
## 0. Load, provenance gates, evaluation sets

Everything must come from the same S2 / Step 2 / Step 3 / Step 4 run: S2 fingerprint, embedding and descriptor SHA-256 (Step 2 manifest), pair-file SHA-256 (Step 3 manifest, stamped into every checkpoint by Step 4). A mismatch stops the notebook.

In [ ]:
EXPECTED_S2_FINGERPRINT = "47d9c2a2fff63eb1"
aop = pd.read_csv(f"{PROC}/AOP-BenchPos_S2.csv")
desc = pd.read_csv(f"{PROC}/descriptors.csv")
need = {"sequence", "labels", "sample_weight", "lso_fold", "cluster_40", "homology_group", "strict_evidence", "homology_resolvable", "source_group"}
assert need <= set(aop.columns) and aop["sequence"].is_unique and desc["sequence"].is_unique
assert set(aop["sequence"]) == set(desc["sequence"])
desc = desc.set_index("sequence").loc[aop["sequence"]].reset_index()
S2_FP = s2_fingerprint(aop)
m2 = json.load(open(f"{PROC}/step2_manifest.json")); pm = json.load(open(f"{PROC}/pairs_manifest.json"))
assert S2_FP == m2["s2_fingerprint"] == pm["s2_fingerprint"], "S2 / Step 2 / Step 3 fingerprints disagree"
assert S2_FP == EXPECTED_S2_FINGERPRINT, f"S2 fingerprint {S2_FP} is not the validated Step 1 build v3"
assert pm.get("build", "").startswith("v4")
assert sha256_file(f"{PROC}/descriptors.csv") == m2["sha256"]["descriptors.csv"]
MODES, PRIMARY_MODE = pm["modes"], pm["primary_mode"]
for key, h in pm["sha256"].items():
    assert sha256_file(f"{PROC}/pairs/{key}") == h, f"pairs/{key} differs from the file Step 3 recorded"
print("S2", S2_FP, "| descriptors + 12 pair files agree with the manifests | modes:", list(MODES), "| primary:", PRIMARY_MODE)

SCALES = {"35M": 480, "150M": 640, "650M": 1280, "3B": 2560}
ESM = {}
for sc, dim in SCALES.items():
    a = np.load(f"{EMB}/esm2_{sc}.npy")
    assert sha256_file(f"{EMB}/esm2_{sc}.npy") == m2["sha256"][f"esm2_{sc}.npy"], f"esm2_{sc}.npy differs from Step 2's record"
    assert a.shape == (len(aop), dim) and np.isfinite(a).all()
    assert open(f"{EMB}/esm2_{sc}_sequences.txt").read().splitlines() == aop["sequence"].tolist()
    ESM[sc] = dict(zip(aop["sequence"], a))
AA = "ACDEFGHIKLMNPQRSTVWY"
AAC = {s: np.array([s.count(a) / len(s) for a in AA]) for s in aop["sequence"]}
DESC = dict(zip(desc["sequence"], desc[[c for c in desc.columns if c != "sequence"]].values))
DCOLS = [c for c in desc.columns if c != "sequence"]
LENGTH = {s: len(s) for s in aop["sequence"]}

mech = aop[aop["labels"].str.contains("FRS|MC")].reset_index(drop=True)
assert mech["lso_fold"].notna().all() and mech["homology_group"].notna().all()
for col in ("sequence", "cluster_40", "homology_group"):
    assert mech.groupby(col)["lso_fold"].nunique().max() == 1, f"{col} spans both folds"
gen_seqs = aop[aop["labels"] == "GEN"]["sequence"].tolist()

def mode_pool(mode):
    d = mech
    if MODES[mode]["strict"]: d = d[d["strict_evidence"].astype(bool)]
    if MODES[mode]["ge4"]:    d = d[d["homology_resolvable"].astype(bool)]
    return d.reset_index(drop=True)
POOL = {m: mode_pool(m) for m in MODES}
EVAL = {}
for m, d in POOL.items():
    q, g = d[d.lso_fold == 1].reset_index(drop=True), d[d.lso_fold == 0].reset_index(drop=True)
    for col in ("sequence", "cluster_40", "homology_group"):
        assert not (set(q[col]) & set(g[col])), f"{m}: query/gallery share {col}"
    EVAL[m] = {"query": q, "gallery": g}
rows = []
for m, e in EVAL.items():
    for part in ("query", "gallery"):
        vc = e[part]["labels"].value_counts()
        rows.append({"mode": m, "set": part, "n": len(e[part]), "FRS": int(vc.get("FRS", 0)), "FRS;MC": int(vc.get("FRS;MC", 0)), "MC": int(vc.get("MC", 0))})
print(pd.DataFrame(rows).to_string(index=False))

---
## 1. Inputs of the release and their provenance

The Step 5 (and, if present, Step 6) result tables are read from `results/`; the release summary is generated from them. Missing mandatory tables stop the notebook.

In [ ]:
MANDATORY = ["step5_mc_retrieval.csv", "step5_paired_comparisons.csv", "step5_probe.csv", "step5_probe_paired.csv", "step5_geometry_heldout.csv"]
OPTIONAL = ["step6_seed_spread.csv", "step6_scale_sweep.csv", "step6_reverse.csv", "step6_meta.json", "step4_collapse_report.csv", "step4_saved_models.csv"]
for f in MANDATORY: assert os.path.exists(f"{RES}/{f}"), f"{f} missing -> run Step 5 first"
R5 = {f: pd.read_csv(f"{RES}/{f}") for f in MANDATORY}
R_OPT = {f: (pd.read_csv(f"{RES}/{f}") if f.endswith(".csv") else json.load(open(f"{RES}/{f}"))) for f in OPTIONAL if os.path.exists(f"{RES}/{f}")}
print("Step 5 tables:", {k: v.shape for k, v in R5.items()})
print("optional tables present:", sorted(R_OPT))
S1_FILES = [f for f in ("AOP-BenchPos_S2.csv", "AOP-Provenance.csv", "AOP-Conflicts.csv", "AOP-Inactive.csv", "excluded_by_primary_evidence.csv", "descriptors.csv",
                        "step1_manifest.json", "step2_manifest.json", "pairs_manifest.json") if os.path.exists(f"{PROC}/{f}")]
print("data files to release:", S1_FILES)

---
## 2. The probes: fit, export as plain arrays, out-of-fold scores

Feature set: ESM-2 650M embedding (1,280 values). Model: standardisation + logistic regression (`C = 1`, balanced class weights) — exactly the Step 5 probe. Two probes are exported:

| probe | training pool | n (MC-active / FRS-only) |
|---|---|---|
| `strict` | peptides with strict per-peptide evidence | 669 (104 / 565) |
| `extended` | all mechanism-active peptides (candidate evidence included) | 714 (149 / 565) |

They are stored as **NumPy arrays** (mean, scale, coefficients, intercept) — no pickle, no scikit-learn version dependence. For every benchmark peptide the **out-of-fold** score from the repeated grouped CV (whole `homology_group`s in 5 folds, **10 different fold assignments**, scores averaged over the 10 repeats; the same fold code as Step 5 v4) is exported too: a score for a peptide the probe was trained on is optimistic and must not be used to judge the probe.

In [ ]:
def group_folds(y, g, k, seed):
    # whole homology groups -> k folds; groups that contain positives balance the positives, the rest balance the size (same code as Step 5 v4)
    rng = np.random.RandomState(seed); ug = np.unique(g); rng.shuffle(ug)
    pos = {u: int(y[g == u].sum()) for u in ug}; tot = {u: int((g == u).sum()) for u in ug}
    ug = sorted(ug, key=lambda u: -(pos[u] * 10_000 + tot[u]))
    fp, fn = np.zeros(k), np.zeros(k); where = {}
    for u in ug:
        j = int(np.lexsort((fn, fp))[0]) if pos[u] > 0 else int(np.argmin(fn))
        where[u] = j; fp[j] += pos[u]; fn[j] += tot[u]
    f = np.array([where[x] for x in g])
    return [(np.where(f != j)[0], np.where(f == j)[0]) for j in range(k)]

PROBES, OOF_TABLE = {}, aop[["sequence", "labels", "lso_fold", "homology_group", "strict_evidence"]].copy()
for mode in ("strict", "extended"):
    d = POOL[mode]; seqs = d["sequence"].tolist(); y = d["labels"].str.contains("MC").astype(int).values; g = d["homology_group"].values.astype(int)
    X = np.stack([ESM["650M"][s] for s in seqs])
    SP = [group_folds(y, g, 5, 1000 + r) for r in range(10)]          # identical to Step 5 v4 (own fold code, independent of the scikit-learn release)
    O = np.full((10, len(y)), np.nan); FA = np.full((10, 5), np.nan)
    for r, folds in enumerate(SP):
        for k, (tr, te) in enumerate(folds):
            sc = StandardScaler().fit(X[tr]); clf = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(sc.transform(X[tr]), y[tr])
            O[r, te] = clf.predict_proba(sc.transform(X[te]))[:, 1]; FA[r, k] = roc_auc_score(y[te], O[r, te])
    assert np.isfinite(O).all()
    oof = O.mean(0); auc = float(np.nanmean(FA)); auc_pooled = float(roc_auc_score(y, oof))
    ref = R5["step5_probe.csv"]; ref = ref[(ref["mode"] == mode) & (ref["features"] == "ESM-2 650M") & ref["cv"].str.startswith("repeated grouped")]["AUROC"].iloc[0]
    assert abs(auc - ref) < 1e-9, f"{mode}: fold-mean out-of-fold AUROC {auc:.6f} differs from Step 5 ({ref:.6f})"
    X = X.astype(np.float64)                                    # final fit and the NumPy re-implementation both in float64 (the out-of-fold run above used the same float32 input as Step 5)
    sc = StandardScaler().fit(X); clf = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(sc.transform(X), y)
    np.savez(f"{BASE}/results/_probe_{mode}.npz", mean=sc.mean_, scale=sc.scale_, coef=clf.coef_[0], intercept=clf.intercept_, n=len(y), n_mc=int(y.sum()))
    z = (X - sc.mean_) / sc.scale_; mine = 1 / (1 + np.exp(-(z @ clf.coef_[0] + clf.intercept_[0])))
    assert np.abs(mine - clf.predict_proba(sc.transform(X))[:, 1]).max() < 1e-9, "NumPy probe differs from scikit-learn"
    PROBES[mode] = {"file": f"{BASE}/results/_probe_{mode}.npz", "n": len(y), "n_mc": int(y.sum()), "oof_auroc": float(auc), "oof_auroc_pooled_of_averaged_score": auc_pooled}
    OOF_TABLE[f"oof_mc_score_{mode}"] = OOF_TABLE["sequence"].map(dict(zip(seqs, oof)))
    print(f"{mode:<9} n={len(y)} MC-active={int(y.sum())}  repeated-grouped-CV fold-mean AUROC={auc:.4f} (= Step 5 v4; AUROC of the averaged OOF score {auc_pooled:.4f})  | NumPy inference equals scikit-learn (max diff < 1e-9)")
OOF_TABLE.to_csv(f"{BASE}/results/_oof_probe_scores.csv", index=False)

---
## 3. The helper module

`aop_procon.py` is written into the release folder as plain text and then **imported back and tested**: sequence validation, composition, the cached-embedding path, the NumPy probe, the contract table, and the ESM-2 pooling rule (max-pool for < 10 aa, mean-pool for ≥ 10 aa, last layer; tested with a stub model, because the real weights are not downloaded here — set `RUN_ESM_SMOKETEST = True` on Colab to compare real embeddings with the cache).

In [ ]:
MODULE_SRC = r'''"""aop_procon -- helper for the AOP-ProCon release v1 (Component 1).

Scope: validate peptide sequences, embed them with ESM-2 650M (same pooling rule as the pipeline), score them with two WEAK
MC-active-vs-FRS-only probes, and return the table described in CONTRACT.md.  The probes are NOT activity predictors.
"""
import os, json
import numpy as np
import pandas as pd

AA = "ACDEFGHIKLMNPQRSTVWY"
HERE = os.path.dirname(os.path.abspath(__file__))
MODEL_NAME = "esm2_t33_650M_UR50D"
POOLING_RULE = "last layer; max-pool over residues for < 10 aa, mean-pool for >= 10 aa"
TRAIN_LENGTH_RANGE = (3, 49)          # lengths present in the benchmark


def validate_sequence(seq):
    """Return a list of problems (empty list = valid)."""
    problems = []
    if not isinstance(seq, str) or not seq:
        return ["empty or non-string"]
    bad = sorted(set(seq) - set(AA))
    if bad:
        problems.append("non-canonical residues: " + "".join(bad))
    if len(seq) < TRAIN_LENGTH_RANGE[0] or len(seq) > TRAIN_LENGTH_RANGE[1]:
        problems.append(f"length {len(seq)} outside the benchmark range {TRAIN_LENGTH_RANGE}")
    return problems


def composition(seqs):
    """20-D amino-acid composition (rows sum to 1)."""
    return np.array([[s.count(a) / len(s) for a in AA] for s in seqs])


def load_benchmark():
    return pd.read_csv(os.path.join(HERE, "data", "AOP-BenchPos_S2.csv"))


def load_cache(scale="650M"):
    """Dictionary sequence -> embedding for the 1,389 benchmark sequences (row order file is checked)."""
    arr = np.load(os.path.join(HERE, "embeddings", f"esm2_{scale}.npy"))
    seqs = open(os.path.join(HERE, "embeddings", f"esm2_{scale}_sequences.txt")).read().splitlines()
    assert len(seqs) == arr.shape[0]
    return dict(zip(seqs, arr))


_PROBES = {}


def load_probe(mode="strict"):
    if mode not in _PROBES:
        p = np.load(os.path.join(HERE, "probes", f"probe_{mode}.npz"))
        _PROBES[mode] = {k: p[k] for k in p.files}
    return _PROBES[mode]


def probe_score(E, mode="strict"):
    """Probability-like score for MC-active (1) vs FRS-only (0). Weak: see CONTRACT.md for the cross-validated AUROC."""
    p = load_probe(mode)
    z = (np.asarray(E, float) - p["mean"]) / p["scale"]
    return 1.0 / (1.0 + np.exp(-(z @ p["coef"] + float(p["intercept"][0]))))


def esm_embed(seqs, model=None, alphabet=None, device="cpu", batch_size=32):
    """ESM-2 650M embeddings with the pipeline's pooling rule. Loads the model through fair-esm unless one is passed in."""
    import torch
    if model is None:
        import esm
        model, alphabet = getattr(esm.pretrained, MODEL_NAME)()
        model = model.to(device)
    model.eval()
    bc = alphabet.get_batch_converter(); nl = model.num_layers; out = []
    for start in range(0, len(seqs), batch_size):
        chunk = list(seqs[start:start + batch_size])
        _, _, tokens = bc([(str(i), s) for i, s in enumerate(chunk)])
        with torch.no_grad():
            reps = model(tokens.to(device), repr_layers=[nl])["representations"][nl]
        for i, s in enumerate(chunk):
            per_res = reps[i, 1:len(s) + 1].cpu()
            out.append((per_res.max(0).values if len(s) < 10 else per_res.mean(0)).numpy())
    return np.stack(out)


def score_peptides(seqs, embeddings=None, device="cpu"):
    """Return the contract table (one row per input sequence, same order).

    embeddings: optional array (n, 1280) aligned with seqs; otherwise the release cache is used for benchmark sequences and
    ESM-2 is run for the others.
    """
    seqs = [s.strip().upper() if isinstance(s, str) else s for s in seqs]
    bench = load_benchmark().set_index("sequence")
    oof = pd.read_csv(os.path.join(HERE, "data", "oof_probe_scores.csv")).set_index("sequence")
    cache = load_cache("650M")
    probs = [validate_sequence(s) for s in seqs]
    if embeddings is None:
        E = np.full((len(seqs), 1280), np.nan)
        need = [i for i, s in enumerate(seqs) if s not in cache and not probs[i]]
        for i, s in enumerate(seqs):
            if s in cache:
                E[i] = cache[s]
        if need:
            E[need] = esm_embed([seqs[i] for i in need], device=device)
    else:
        E = np.asarray(embeddings, float)
        assert E.shape == (len(seqs), 1280), "embeddings must have shape (n, 1280)"
    ok = np.isfinite(E).all(1)
    rows = []
    for i, s in enumerate(seqs):
        mech_active = False
        r = {"sequence": s, "length": len(s) if isinstance(s, str) else None, "valid": not probs[i], "problems": "; ".join(probs[i]),
             "in_benchmark": s in bench.index}
        if r["in_benchmark"]:
            b = bench.loc[s]
            se = b["strict_evidence"]; se = bool(se) if pd.notna(se) else None          # GEN peptides have no strict_evidence value
            r.update({"benchmark_labels": b["labels"], "fold": b["lso_fold"], "homology_group": b["homology_group"], "strict_evidence": se})
            mech_active = b["labels"] != "GEN"
            r["oof_mc_score_strict"] = oof.loc[s].get("oof_mc_score_strict", np.nan)
            r["oof_mc_score_extended"] = oof.loc[s].get("oof_mc_score_extended", np.nan)
        for mode in ("strict", "extended"):
            r[f"mc_score_{mode}"] = float(probe_score(E[i:i + 1], mode)[0]) if ok[i] else np.nan
        # a probe score is in-sample if the peptide belonged to the pool that probe was fitted on
        r["score_is_in_sample_strict"] = bool(r["in_benchmark"] and mech_active and r.get("strict_evidence") is True)
        r["score_is_in_sample_extended"] = bool(r["in_benchmark"] and mech_active)
        r["warning"] = ("weak signal: repeated-grouped-CV AUROC about 0.6-0.7, not distinguishable from amino-acid composition; the single feature source_group is as good, "
                        "and ESM-2 adds nothing measurable on top of composition + length + source; for benchmark peptides use the oof_* columns")
        rows.append(r)
    return pd.DataFrame(rows)
'''

os.makedirs(REL, exist_ok=True)
for sub in ("data", "embeddings", "probes", "models", "results"): os.makedirs(f"{REL}/{sub}", exist_ok=True)
open(f"{REL}/aop_procon.py", "w", encoding="utf-8").write(MODULE_SRC)
# stage the pieces the module reads
for f in ("AOP-BenchPos_S2.csv",): shutil.copy(f"{PROC}/{f}", f"{REL}/data/{f}")
for mode in PROBES: shutil.copy(PROBES[mode]["file"], f"{REL}/probes/probe_{mode}.npz")
shutil.copy(f"{BASE}/results/_oof_probe_scores.csv", f"{REL}/data/oof_probe_scores.csv")
for sc in SCALES:
    shutil.copy(f"{EMB}/esm2_{sc}.npy", f"{REL}/embeddings/"); shutil.copy(f"{EMB}/esm2_{sc}_sequences.txt", f"{REL}/embeddings/")

import importlib, sys, types
sys.path.insert(0, REL); sys.modules.pop("aop_procon", None); import aop_procon as AP; importlib.reload(AP)

# --- tests
assert AP.validate_sequence("ACDEFG") == [] and AP.validate_sequence("ACDXB") and AP.validate_sequence("A") and AP.validate_sequence("")
assert np.allclose(AP.composition(["AAC"]).sum(1), 1.0)
seqs = aop["sequence"].iloc[:50].tolist()
tab = AP.score_peptides(seqs)                                          # cached-embedding path, no model needed
assert len(tab) == 50 and tab["valid"].all() and tab["in_benchmark"].all() and np.isfinite(tab["mc_score_strict"]).all()
Xs = np.stack([ESM["650M"][s] for s in seqs])
assert np.allclose(tab["mc_score_strict"].values, AP.probe_score(Xs, "strict"))
oof_ref = OOF_TABLE.set_index("sequence")
assert np.allclose(tab["oof_mc_score_strict"].fillna(-1).values, oof_ref.loc[seqs, "oof_mc_score_strict"].fillna(-1).values)
bad = AP.score_peptides(["ACDXB", "AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAA"], embeddings=np.zeros((2, 1280)))
assert (~bad["valid"]).all() and bad["problems"].str.len().gt(0).all()
gen_row = AP.score_peptides([aop.loc[aop["labels"] == "GEN", "sequence"].iloc[0]]).iloc[0]
assert gen_row["in_benchmark"] and not gen_row["score_is_in_sample_strict"] and not gen_row["score_is_in_sample_extended"] and gen_row["strict_evidence"] is None
frs_row = AP.score_peptides([aop.loc[aop["labels"] == "FRS", "sequence"].iloc[0]]).iloc[0]
assert frs_row["score_is_in_sample_strict"] and frs_row["score_is_in_sample_extended"]
cand = AP.score_peptides([aop.loc[(aop["labels"] == "MC") & (aop["strict_evidence"] == False), "sequence"].iloc[0]]).iloc[0]
assert (not cand["score_is_in_sample_strict"]) and cand["score_is_in_sample_extended"]
print("module tests passed: validation, composition, cached-embedding path, NumPy probe == module probe, out-of-fold columns, invalid input flagged, in-sample flags (GEN / strict / candidate)")

# --- pooling rule with a stub model (the real weights are not needed to test the rule)
import torch
class _Alpha:
    def get_batch_converter(self):
        def conv(batch):
            L = max(len(s) for _, s in batch); return None, None, torch.zeros(len(batch), L + 2, dtype=torch.long)
        return conv
class _Stub:
    num_layers = 3
    def eval(self): return self
    def __call__(self, tokens, repr_layers):
        torch.manual_seed(0); B, L = tokens.shape; return {"representations": {3: torch.randn(B, L, 1280)}}
test = ["ACD", "ACDEFGHIK", "ACDEFGHIKL", "ACDEFGHIKLMNPQ"]
out = AP.esm_embed(test, model=_Stub(), alphabet=_Alpha(), batch_size=4)
torch.manual_seed(0); reps = torch.randn(4, 16, 1280)
for i, s in enumerate(test):
    pr = reps[i, 1:len(s) + 1]; exp = (pr.max(0).values if len(s) < 10 else pr.mean(0)).numpy()
    assert np.allclose(out[i], exp, atol=1e-6), f"pooling wrong for length {len(s)}"
print("pooling rule verified: max-pool < 10 aa, mean-pool >= 10 aa")

SMOKE_NOTE = "A fresh-vs-cached ESM-2 650M comparison was **not** run for this release (`RUN_ESM_SMOKETEST = False`)."
if RUN_ESM_SMOKETEST:
    get_ipython().system("pip install -q fair-esm")
    chk = AP.esm_embed(seqs[:5]); dmax = float(np.abs(chk - Xs[:5]).max()); cos = float(((chk * Xs[:5]).sum(1) / (np.linalg.norm(chk, axis=1) * np.linalg.norm(Xs[:5], axis=1))).min())
    print(f"ESM smoke test: max |fresh - cached| = {dmax:.2e}, min cosine = {cos:.6f}"); assert cos > 0.99999
    SMOKE_NOTE = f"A fresh-vs-cached comparison of 5 sequences with ESM-2 650M gave max |difference| {dmax:.1e} and minimum cosine {cos:.6f}."
else:
    print("(RUN_ESM_SMOKETEST = False: the real ESM-2 weights were not downloaded; set it to True on Colab to compare fresh and cached embeddings)")

---
## 4. Documents, generated from the result tables

`CONTRACT.md`, `README.md` and `RESULTS_SUMMARY.md` are written by this cell. Every number in them is read from the Step 5 / Step 6 tables, so they cannot disagree with the notebooks.

In [ ]:
def md_table(df, cols, fmt=None):
    fmt = fmt or {}
    head = "| " + " | ".join(cols) + " |\n|" + "|".join("---" for _ in cols) + "|\n"
    body = ""
    for _, r in df.iterrows():
        body += "| " + " | ".join((fmt[c](r[c]) if c in fmt and pd.notna(r[c]) else (f"{r[c]:.3f}" if isinstance(r[c], (float, np.floating)) and pd.notna(r[c]) else str(r[c]))) for c in cols) + " |\n"
    return head + body

pr = R5["step5_probe.csv"]; grp = pr[pr["cv"].str.startswith("repeated grouped")]; lk = pr[pr["cv"].str.startswith("LEAKY")]
def auc_txt(mode, feat):
    r = grp[(grp["mode"] == mode) & (grp["features"] == feat)].iloc[0]; return f"{r['AUROC']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]"
def nb_txt(mode):
    r = grp[(grp["mode"] == mode) & grp["features"].str.startswith("NULL")].iloc[0]; return f"{r['AUROC']:.3f} [{r['lo']:.3f}, {r['hi']:.3f}]"
def rr(mode, feat):
    r = grp[(grp["mode"] == mode) & (grp["features"] == feat)].iloc[0]; return f"{r['AUROC_rep_min']:.3f}-{r['AUROC_rep_max']:.3f}"
pairedP = R5["step5_probe_paired.csv"]
def dlt(mode, a, b="AAC-20"):
    r = pairedP[(pairedP["mode"] == mode) & (pairedP["A"] == a) & (pairedP["B"] == b)].iloc[0]; return f"{r['delta_AUROC']:+.3f} [{r['ci_lo']:+.3f}, {r['ci_hi']:+.3f}] ({r['verdict']})"
mr = R5["step5_mc_retrieval.csv"]; mrP = mr[mr["mode"] == PRIMARY_MODE]
best_base = mrP[mrP["kind"] == "baseline"].sort_values("AP", ascending=False).iloc[0]
tr_models = mrP[mrP["kind"] == "model"]
CH_AP = float(mrP["chance_AP"].iloc[0]); CH_R5 = float(mrP["chance_R@5"].iloc[0])
pc = R5["step5_paired_comparisons.csv"]; vs_raw = pc[pc["A"].isin(tr_models["representation"]) & pc["B"].isin(["RAW 650M", "RAW 150M"]) & (pc["mode"] == PRIMARY_MODE)]
n_worse = int(((vs_raw["delta_AP"] < 0) & (vs_raw["verdict"] != "no difference detected")).sum())

CONTRACT = f'''# AOP-ProCon (Component 1) — integration contract, release v1

**Benchmark version:** S2 build v3, fingerprint `{S2_FP}` ({len(aop)} unique peptides, {len(mech)} mechanism-active).  **Embedding model:** ESM-2 650M (`esm2_t33_650M_UR50D`), {AP.POOLING_RULE}.

## Input
* A list / column of peptide sequences: upper-case, the 20 canonical amino acids only, length {AP.TRAIN_LENGTH_RANGE[0]}–{AP.TRAIN_LENGTH_RANGE[1]} (the range present in the benchmark; other lengths are accepted but flagged).
* Python: `aop_procon.score_peptides(list_of_sequences)`; the optional argument `embeddings` takes a precomputed `(n, 1280)` array aligned with the input.

## Output — one row per input peptide, same order (`pandas.DataFrame`, also writable as CSV)
| column | type | meaning |
|---|---|---|
| `sequence`, `length` | str, int | the input |
| `valid`, `problems` | bool, str | `False` if non-canonical residues or empty; length outside {AP.TRAIN_LENGTH_RANGE} is reported in `problems` but the row is still scored |
| `in_benchmark` | bool | the sequence is one of the {len(aop)} benchmark peptides |
| `benchmark_labels`, `fold`, `homology_group`, `strict_evidence` | str, float, float, bool | only if `in_benchmark`: `FRS` / `MC` / `FRS;MC` / `GEN` (GEN = mechanism **unknown**, not inactive; `strict_evidence` is `None` for `GEN`) |
| `mc_score_strict`, `mc_score_extended` | float in (0, 1) | probability-like score for "MC-active" vs "FRS-only" from the two probes (ESM-2 650M, logistic regression) |
| `oof_mc_score_strict`, `oof_mc_score_extended` | float | **out-of-fold** score (grouped by `homology_group`) for benchmark peptides — use these, not the `mc_score_*` columns, to judge the probe on benchmark data |
| `score_is_in_sample_strict`, `score_is_in_sample_extended` | bool | `True` if the peptide was in the pool that probe was fitted on (mechanism-active; for `strict` also `strict_evidence`), so the matching `mc_score_*` is optimistic. `GEN` peptides are in neither pool |
| `warning` | str | fixed text: the signal is weak |

Embeddings (`embeddings/esm2_{{35M,150M,650M,3B}}.npy`, row order in the `*_sequences.txt` files) are the other usable output; they are frozen ESM-2 vectors, not a trained projection.

**Provenance of the cached embeddings** (`step2_manifest.json`: {m2["embedding_source"]}). Files marked "already aligned" or "re-aligned" were reused from earlier extraction runs whose checkpoint, layer and pooling settings were not recorded; "extracted" files were produced by the Step 2 code. {SMOKE_NOTE}

## Performance that may be quoted (repeated homology-grouped CV: 5 folds × 10 fold assignments, fold-mean AUROC, MC-active vs FRS-only)
| probe | n (MC-active / FRS-only) | ESM-2 650M AUROC [95 % CI] (range over the 10 repeats) | amino-acid composition | **source_group only** | composition + length + source | paired ΔAUROC: ESM-2 650M − composition | ESM-2 650M − source only | ESM-2 650M + (comp. + length + source) − (comp. + length + source) |
|---|---|---|---|---|---|---|---|---|
| `strict` | {PROBES["strict"]["n"]} ({PROBES["strict"]["n_mc"]} / {PROBES["strict"]["n"] - PROBES["strict"]["n_mc"]}) | {auc_txt("strict", "ESM-2 650M")} ({rr("strict", "ESM-2 650M")}) | {auc_txt("strict", "AAC-20")} | {auc_txt("strict", "SOURCE")} | {auc_txt("strict", "AAC-20 + LENGTH + SOURCE")} | {dlt("strict", "ESM-2 650M")} | {dlt("strict", "ESM-2 650M", "SOURCE")} | {dlt("strict", "ESM-2 650M + AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH + SOURCE")} |
| `extended` | {PROBES["extended"]["n"]} ({PROBES["extended"]["n_mc"]} / {PROBES["extended"]["n"] - PROBES["extended"]["n_mc"]}) | {auc_txt("extended", "ESM-2 650M")} ({rr("extended", "ESM-2 650M")}) | {auc_txt("extended", "AAC-20")} | {auc_txt("extended", "SOURCE")} | {auc_txt("extended", "AAC-20 + LENGTH + SOURCE")} | {dlt("extended", "ESM-2 650M")} | {dlt("extended", "ESM-2 650M", "SOURCE")} | {dlt("extended", "ESM-2 650M + AAC-20 + LENGTH + SOURCE", "AAC-20 + LENGTH + SOURCE")} |

Chance reference = fold-mean AUROC of pure-noise features over 20 noise seeds: strict {nb_txt("strict")}, extended {nb_txt("extended")}. The CIs are group bootstraps with the fitted models held fixed; they are narrower than the true uncertainty.

**Confounds (read before quoting any number above).** Every FRS-only peptide in the benchmark comes from AnOxPePred, while {int(round(100 * aop[aop["strict_evidence"] == True]["source_group"].eq("external_literature").sum() / max(1, aop[(aop["strict_evidence"] == True) & aop["labels"].str.contains("MC")].shape[0])))} % of the strict MC-active peptides come from external literature, so the single feature `source_group` already separates the two classes at least as well as ESM-2 650M (table above). Length acts through the share of tripeptides (FRS-only 43 %, MC-active 16 % in `strict`), not through the mean length (5.75 vs 5.86 aa). The probe scores therefore **cannot** be read as mechanism evidence; with source and length controlled, ESM-2 adds nothing measurable (paired ΔAUROC CIs include 0), and in a single-source, length-matched subset every representation falls to AUROC ≈ 0.58-0.62 (STEP 08).

A **single** `GroupKFold` split moved the strict ESM-2 650M AUROC between 0.59 and 0.66 depending on the scikit-learn release (the value 0.593 quoted in earlier builds is one such split); the pooled out-of-fold AUROC is biased by fold-specific intercepts. Use the fold-mean values above.

With random (leaky) folds the same probe reaches {lk[(lk["mode"] == "strict") & (lk["features"] == "ESM-2 650M")]["AUROC"].iloc[0]:.3f} (strict) and {lk[(lk["mode"] == "extended") & (lk["features"] == "ESM-2 650M")]["AUROC"].iloc[0]:.3f} (extended): **never quote an AUROC from a split that is not grouped by `homology_group`, nor from a single split.**

## What the other components must not assume
1. The probes are **not** antioxidant-activity, toxicity or safety predictors; they only separate MC-active from FRS-only labels, weakly.
2. `GEN` means "antioxidant mechanism not established", not "inactive". The 27 peptides in `data/AOP-Inactive.csv` with `db_conflict == False` are the only clean negatives.
3. Benchmark labels combine assay-level and database-level evidence; `strict_evidence == True` marks per-peptide evidence only.
4. The trained contrastive heads in `models/` are research artefacts (worse than the untrained baselines); do not load them for scoring.
5. Scores outside {AP.TRAIN_LENGTH_RANGE[0]}–{AP.TRAIN_LENGTH_RANGE[1]} aa, or for non-plant-derived sequences, are extrapolation.
6. **Source and length are confounded with the label** (see the performance section): a model can learn "which database / how many tripeptides" instead of "which mechanism". New benchmark peptides from other sources are outside what the probes were validated on.

## Versioning
`MANIFEST.json` lists the SHA-256 of every file and the S2 / embedding / pair-file fingerprints. A change of any input changes the fingerprint; consumers should assert it.
'''

README = f'''# AOP-ProCon — Component 1 release v1

Leak-controlled multi-label benchmark of plant antioxidant peptides (free-radical scavenging FRS, metal chelation MC, unknown mechanism GEN), ESM-2 embeddings, two weak supervised probes, and the evaluation that shows what does and does not work.

## Layout
```
README.md  CONTRACT.md  RESULTS_SUMMARY.md  MANIFEST.json  aop_procon.py
data/        AOP-BenchPos_S2.csv, provenance / conflicts / inactive tables, descriptors.csv, pairs/<mode>/..., manifests, oof_probe_scores.csv
embeddings/  esm2_{{35M,150M,650M,3B}}.npy  + *_sequences.txt (row order)
probes/      probe_strict.npz, probe_extended.npz   (NumPy arrays: mean, scale, coef, intercept)
models/      Step 4 contrastive checkpoints (research artefacts only)
results/     Step 4-6 result tables
```

## Quick start
```python
import aop_procon as ap
table = ap.score_peptides(["FLQL", "YPWT", "DHTK"])      # cached embedding if in the benchmark, otherwise ESM-2 (needs fair-esm + torch)
print(table[["sequence", "valid", "in_benchmark", "mc_score_strict", "warning"]])
```
Python ≥ 3.9, numpy, pandas; `torch` + `fair-esm` only for sequences outside the benchmark.

## Reproduction (Google Colab, `BASE = /content/drive/MyDrive/AOP-ProCon`)
`STEP01` (dataset, fingerprint `{S2_FP}`) → `STEP02` (descriptors, embeddings) → `STEP03` (pair sets, three modes) → `STEP04` (15 contrastive models) → `STEP05` (evaluation) → `STEP06` (seed spread, scale sweep, reverse direction) → `STEP07` (this package). Each step asserts the fingerprints / SHA-256 of the previous one.

## Headline findings (details and tables in RESULTS_SUMMARY.md)
* Probe AUROC (repeated homology-grouped CV, fold-mean): strict {auc_txt("strict", "ESM-2 650M")}, extended {auc_txt("extended", "ESM-2 650M")} — not distinguishable from amino-acid composition, and the single feature `source_group` gets {auc_txt("strict", "SOURCE")} (strict) / {auc_txt("extended", "SOURCE")} (extended); ESM-2 adds nothing measurable on top of composition + length + source (see CONTRACT.md).
* Random folds inflate the same probe to {lk[(lk["mode"] == "strict") & (lk["features"] == "ESM-2 650M")]["AUROC"].iloc[0]:.2f} / {lk[(lk["mode"] == "extended") & (lk["features"] == "ESM-2 650M")]["AUROC"].iloc[0]:.2f}: leakage must be controlled.
* The contrastive projection is worse than the untrained ESM-2 embedding in {n_worse} of {len(vs_raw)} comparisons (paired CI excludes 0); the best untrained baseline is `{best_base["representation"]}` (AP {best_base["AP"]:.3f}, chance {CH_AP:.3f}).

## Limitations
One S2 build, one pairing rule, one loss; tens of MC-active queries; noisy assay-level labels; no external validation set; in the `strict` split all 18 MC-only peptides lie in one fold.
'''

def _opt_block(name, title, cols):
    if name not in R_OPT: return f"\n## {title}\n_(not run: `{name}` not found — run Step 6)_\n"
    return f"\n## {title}\n" + md_table(R_OPT[name], [c for c in cols if c in R_OPT[name].columns]) + "\n"

SUMMARY = (f"# Results summary (generated from results/*.csv)\n\nPrimary mode `{PRIMARY_MODE}`; train fold 0 → evaluate fold 1; MC-active queries; chance AP {CH_AP:.3f}, chance Recall@5 {CH_R5:.3f}.\n\n"
           "## MC retrieval (Step 5)\n" + md_table(mrP.sort_values("AP", ascending=False), ["representation", "kind", "n_queries", "AP", "AP_lo", "AP_hi", "lift", "R@5", "mean_cos_heldout"]) +
           "\n## Paired comparisons (Step 5, Holm-corrected)\n" + md_table(pc, ["comparison", "mode", "n", "delta_AP", "ci_lo", "ci_hi", "p_holm", "verdict"]) +
           "\n## Supervised probe, repeated grouped CV vs leaky folds (Step 5 v4; fold-mean AUROC)\n" + md_table(pr, ["mode", "features", "n", "AUROC", "lo", "hi", "AUROC_rep_min", "AUROC_rep_max", "cv"]) +
           _opt_block("step6_seed_spread.csv", "Seed spread (Step 6)", ["model", "n", "AP_mean", "AP_sd", "AP_min", "AP_max", "dAP_mean", "dAP_sd", "seeds_below_raw", "raw_AP"]) +
           _opt_block("step6_scale_sweep.csv", "Scale sweep (Step 6)", ["scale", "model", "raw_AP", "AP_mean", "AP_sd", "dAP_mean", "seeds_below_raw"]) +
           _opt_block("step6_reverse.csv", "Reverse fold direction (Step 6)", ["model", "n", "AP_mean", "AP_sd", "dAP_mean", "seeds_below_raw", "raw_AP", "chance_AP"]))
for name, text in (("CONTRACT.md", CONTRACT), ("README.md", README), ("RESULTS_SUMMARY.md", SUMMARY)):
    open(f"{REL}/{name}", "w", encoding="utf-8").write(text)
print("written:", ["CONTRACT.md", "README.md", "RESULTS_SUMMARY.md"], "| contract length", len(CONTRACT), "chars")
print("\n" + CONTRACT[:1800])

---
## 5. Assemble, hash, archive, read back

In [ ]:
for f in S1_FILES: shutil.copy(f"{PROC}/{f}", f"{REL}/data/{f}")
shutil.copytree(f"{PROC}/pairs", f"{REL}/data/pairs", dirs_exist_ok=True)
n_models = 0
for f in sorted(os.listdir(RES)):
    if f.startswith("model_") and f.endswith(".pt"): shutil.copy(f"{RES}/{f}", f"{REL}/models/{f}"); n_models += 1
open(f"{REL}/models/README.txt", "w").write("Contrastive projection heads from Step 4 (research artefacts, reproducibility only).\nThey are worse than the untrained ESM-2 embedding in Step 5; do not use them for scoring.\n")
for f in sorted(os.listdir(RES)):
    if (f.startswith("step4_") or f.startswith("step5_") or f.startswith("step6_")) and not f.startswith("_") and f != "step5_probe.png":
        shutil.copy(f"{RES}/{f}", f"{REL}/results/{f}")
print(f"copied {n_models} checkpoints and the result tables")

files = sorted(os.path.relpath(os.path.join(d, f), REL).replace(os.sep, "/") for d, _, fs in os.walk(REL) for f in fs if f not in ("MANIFEST.json",) and "__pycache__" not in d)
manifest = {"release": RELEASE_NAME, "date": str(datetime.date.today()), "s2_fingerprint": S2_FP, "primary_mode": PRIMARY_MODE,
            "step2_manifest_sha256": m2["sha256"], "pairs_manifest_sha256": pm["sha256"], "versions": VERSIONS,
            "probes": {k: {"n": v["n"], "n_mc": v["n_mc"], "repeated_grouped_cv_foldmean_auroc_esm650M": round(v["oof_auroc"], 6), "cv_design": "5 folds x 10 fold assignments on homology_group, fold-mean AUROC, C=1 logistic regression"} for k, v in PROBES.items()},
            "files": {f: sha256_file(f"{REL}/{f}") for f in files}}
json.dump(manifest, open(f"{REL}/MANIFEST.json", "w"), indent=2)
zpath = f"{BASE}/release/{RELEASE_NAME}.zip"
with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
    for f in files + ["MANIFEST.json"]: z.write(f"{REL}/{f}", f"{RELEASE_NAME}/{f}")
print(f"{len(files)} files hashed; archive {zpath} = {os.path.getsize(zpath) / 1e6:.1f} MB")

# --- read back from the archive: every hash, the fingerprint, the probe, the module
chk = f"{BASE}/release/_verify"; shutil.rmtree(chk, ignore_errors=True)
with zipfile.ZipFile(zpath) as z: z.extractall(chk)
R2 = f"{chk}/{RELEASE_NAME}"; mf = json.load(open(f"{R2}/MANIFEST.json"))
bad = [f for f, h in mf["files"].items() if sha256_file(f"{R2}/{f}") != h]
assert not bad, f"hash mismatch after archiving: {bad[:3]}"
assert mf["s2_fingerprint"] == S2_FP == s2_fingerprint(pd.read_csv(f"{R2}/data/AOP-BenchPos_S2.csv"))
sys.path.insert(0, R2); sys.modules.pop("aop_procon", None); import aop_procon as AP2
t2 = AP2.score_peptides(aop["sequence"].iloc[:10].tolist()); assert np.allclose(t2["mc_score_strict"], AP.score_peptides(aop["sequence"].iloc[:10].tolist())["mc_score_strict"])
sys.path.remove(R2); shutil.rmtree(chk, ignore_errors=True)
print("read-back OK: all file hashes, S2 fingerprint, the module and the probe work from the extracted archive")
shutil.rmtree(f"{REL}/__pycache__", ignore_errors=True)
print("\nrelease folder:", REL)
for f in sorted(os.listdir(REL)): print("  ", f)

---
## Summary

Open `README.md`, `CONTRACT.md` and `RESULTS_SUMMARY.md` in the release folder. Hand `aop_procon_v1.zip` to the other components together with the contract; the MANIFEST lets them verify what they received.